[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/05_Subgraphs_Tests_Loops/Subgraphs_Tests_Loops_Deep_Dive.ipynb)

# 1.5 Subgraphs: Tests and Loops — Deep Dive

ONNX supports **control flow** through operators that take entire sub-graphs as attributes. This enables conditional execution, iteration, and scan patterns within a static computation graph.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [Control Flow in Static Graphs](#section-1) | Why subgraphs are needed |
| 2 | [The If Operator](#section-2) | Conditional branching with then/else subgraphs |
| 3 | [Building an If Model](#section-3) | Step-by-step code example |
| 4 | [The Scan Operator](#section-4) | Iterating over tensor rows |
| 5 | [Building a Scan Model](#section-5) | Cumulative sum with Scan |
| 6 | [The Loop Operator](#section-6) | For/while loop semantics |
| 7 | [Control Flow vs Vectorized Ops](#section-7) | When to avoid control flow |
| 8 | [Formal Semantics](#section-8) | Mathematical formulation |
| 9 | [Key Takeaways & Interview Questions](#section-9) | Summary |

### Prerequisites

- Completed **1.1–1.4** (graph construction, serialization, initializers, opsets)
- Understanding of DAG execution model

<a id='section-1'></a>
## Section 1: Control Flow in Static Graphs

### The Challenge

ONNX computation graphs are **DAGs** — directed acyclic graphs. By definition, a DAG has no cycles, which means straightforward iteration is impossible. And since all nodes in a DAG execute unconditionally (in topological order), conditional branching is also impossible.

Yet many real models need control flow:

| Pattern | Example | Why Needed |
|---------|---------|------------|
| **Conditional** | Skip expensive computation if confidence > threshold | Efficiency |
| **Iteration** | Process sequence elements one at a time (RNN) | Sequential dependencies |
| **Early exit** | Stop decoding when EOS token generated | Variable-length output |

### The Solution: Graphs-as-Attributes

ONNX solves this by allowing certain operators to take **entire GraphProto objects as attributes**. The control flow operator manages when and how many times these subgraphs execute.

```
┌──────────────────────────────────────────────────────────────┐
│                    Main Graph (DAG)                          │
│                                                              │
│   input X ──▶ [Compute condition] ──▶ cond (bool)           │
│                                          │                   │
│                                          ▼                   │
│                                   ┌──────────┐              │
│                                   │    If     │              │
│                                   │          │              │
│                     ┌─────────────┤          ├──────────┐   │
│                     │  then_branch │          │else_branch│  │
│                     │  (GraphProto)│          │(GraphProto)│ │
│                     │  ┌────────┐ │          │┌────────┐ │  │
│                     │  │Subgraph│ │          ││Subgraph│ │  │
│                     │  │  A     │ │          ││  B     │ │  │
│                     │  └────────┘ │          │└────────┘ │  │
│                     └─────────────┘          └──────────┘   │
│                                   │                          │
│                                   ▼                          │
│                              output Y                        │
└──────────────────────────────────────────────────────────────┘
```

### The Three Control Flow Operators

| Operator | Attributes | Semantics |
|----------|-----------|----------|
| `If` | `then_branch`, `else_branch` | Execute one of two subgraphs based on a boolean condition |
| `Loop` | `body` | Repeat a subgraph with loop-carried state |
| `Scan` | `body` | Iterate over rows of input tensors, accumulating outputs |

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install onnx onnxruntime numpy matplotlib

In [ ]:
import numpy as np
import onnx
from onnx import TensorProto
from onnx.helper import (
    make_node, make_graph, make_model,
    make_tensor_value_info, make_opsetid)
from onnx.numpy_helper import from_array
from onnx.checker import check_model
from onnxruntime import InferenceSession

print('Setup complete!')

<a id='section-2'></a>
## Section 2: The If Operator

### Formal Semantics

The `If` operator takes a **boolean scalar** condition and executes one of two subgraphs:

$$\text{If}(c, G_{\text{then}}, G_{\text{else}}) = \begin{cases} \text{eval}(G_{\text{then}}) & \text{if } c = \text{true} \\ \text{eval}(G_{\text{else}}) & \text{if } c = \text{false} \end{cases}$$

### Schema

| Component | Type | Description |
|-----------|------|-------------|
| **Input** `cond` | `tensor(bool)` scalar | The condition to test |
| **Attr** `then_branch` | `GraphProto` | Subgraph executed when `cond=true` |
| **Attr** `else_branch` | `GraphProto` | Subgraph executed when `cond=false` |
| **Outputs** | `tensor(T)` | Must match between both branches (same count, compatible types) |

### Critical Constraints

1. Both branches must produce the **same number of outputs** with **compatible types**
2. The condition must be a **scalar boolean tensor** (rank 0)
3. Subgraphs can access **outer scope tensors** (variables from the parent graph)

### Graph Structure

![If Operator Graph](assets/dot_if_py.png)

<a id='section-3'></a>
## Section 3: Building an If Model

We will build a model that computes:

$$Y = \begin{cases} [1.0] & \text{if } \sum X > 0 \\ [-1.0] & \text{if } \sum X \leq 0 \end{cases}$$

This requires:
1. Computing $\sum X$ with `ReduceSum`
2. Comparing against 0 with `Greater`
3. Branching with `If`

In [ ]:
# ── Build the If model ────────────────────────────────────────────

# Initializer: zero for comparison
zero = from_array(np.array([0], dtype=np.float32), name='zero')

X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

# Main graph nodes: compute condition
rsum = make_node('ReduceSum', ['X'], ['rsum'])
cond = make_node('Greater', ['rsum', 'zero'], ['cond'])

# ── Then branch: return [1.0] ──────────────────────────────────
then_out = make_tensor_value_info('then_out', TensorProto.FLOAT, None)
then_cst = from_array(np.array([1.0]).astype(np.float32))
then_const_node = make_node('Constant', [], ['then_out'],
                            value=then_cst, name='cst_then')
then_body = make_graph([then_const_node], 'then_body', [], [then_out])

# ── Else branch: return [-1.0] ─────────────────────────────────
else_out = make_tensor_value_info('else_out', TensorProto.FLOAT, None)
else_cst = from_array(np.array([-1.0]).astype(np.float32))
else_const_node = make_node('Constant', [], ['else_out'],
                            value=else_cst, name='cst_else')
else_body = make_graph([else_const_node], 'else_body', [], [else_out])

# ── If node ────────────────────────────────────────────────────
if_node = make_node('If', ['cond'], ['Y'],
                    then_branch=then_body,
                    else_branch=else_body)

# ── Assemble ───────────────────────────────────────────────────
graph = make_graph([rsum, cond, if_node], 'if_example', [X], [Y], [zero])
model_if = make_model(graph, opset_imports=[make_opsetid('', 15)])
model_if.ir_version = 8

sess_if = InferenceSession(model_if.SerializeToString(),
                           providers=['CPUExecutionProvider'])

# Test with positive and negative sums
test_cases = [
    ('Positive sum', np.ones((3, 2), dtype=np.float32)),
    ('Negative sum', -np.ones((3, 2), dtype=np.float32)),
    ('Zero input',   np.zeros((2, 2), dtype=np.float32)),
    ('Mixed (net>0)', np.array([[10, -1], [-2, 1]], dtype=np.float32)),
]

print(f'{"Test Case":>20s} | {"sum(X)":>8s} | {"Output":>8s} | {"Branch"}')
print('-' * 60)
for name, x_data in test_cases:
    result = sess_if.run(None, {'X': x_data})[0]
    s = x_data.sum()
    branch = 'then (+1)' if result[0] > 0 else 'else (-1)'
    print(f'{name:>20s} | {s:>8.1f} | {result[0]:>8.1f} | {branch}')

<a id='section-4'></a>
## Section 4: The Scan Operator

### Formal Semantics

The `Scan` operator iterates over the **rows** (or slices along a specified axis) of one or more input tensors, executing a body subgraph at each step. It maintains **state** across iterations and accumulates **scan outputs**.

Formally, for $T$ timesteps:

$$\text{For } t = 0, 1, \ldots, T-1:$$
$$s_{t+1}, y_t = f_{\text{body}}(s_t, x_t)$$

where:
- $s_t$ is the **state** at timestep $t$ (carried across iterations)
- $x_t$ is the **input slice** at timestep $t$
- $y_t$ is the **scan output** at timestep $t$ (accumulated into a tensor)
- $f_{\text{body}}$ is the body subgraph

### Scan vs Loop vs Vectorized

```
Scan operator data flow:

  Input X: ┌─────┬─────┬─────┬─────┐
           │ x₀  │ x₁  │ x₂  │ x₃  │    (rows of X)
           └──┬──┴──┬──┴──┬──┴──┬──┘
              │     │     │     │
  State: s₀──┤     │     │     │
              ▼     │     │     │
         ┌────────┐ │     │     │
    s₁ ◀─┤  body  ├─▶ y₀ │     │
         └────────┘       │     │
              │           │     │
              ▼           │     │
         ┌────────┐       │     │
    s₂ ◀─┤  body  ├─▶ y₁ │     │
         └────────┘       │     │
              │           │     │
              ▼           ▼     │
         ┌────────┐             │
    s₃ ◀─┤  body  ├─▶ y₂       │
         └────────┘             │
              │                 │
              ▼                 ▼
         ┌────────┐
    s₄ ◀─┤  body  ├─▶ y₃
         └────────┘

  Final state: s₄
  Scan output: [y₀, y₁, y₂, y₃]  (stacked)
```

![Scan Operator](assets/scanop.png)

### Schema

| Component | Description |
|-----------|-------------|
| **Inputs (first N)** | Initial state values $s_0$ |
| **Inputs (remaining)** | Tensors to scan over (sliced along axis) |
| **Attr** `body` | Body subgraph: `(state_in, scan_in) → (state_out, scan_out)` |
| **Attr** `num_scan_inputs` | How many inputs are scanned (vs state) |
| **Outputs (first N)** | Final state values $s_T$ |
| **Outputs (remaining)** | Accumulated scan outputs (stacked) |

<a id='section-5'></a>
## Section 5: Building a Scan Model — Cumulative Sum

Let's implement a **cumulative sum** using Scan:

$$s_{t+1} = s_t + x_t, \quad y_t = s_{t+1}$$

For input $X = [1, 2, 3, 4]$, the expected output is $[1, 3, 6, 10]$.

In [ ]:
# ── Cumulative sum with Scan ──────────────────────────────────

# Body subgraph: (state, x_i) → (state + x_i, state + x_i)
state_in = make_tensor_value_info('state_in', TensorProto.FLOAT, [1])
x_i = make_tensor_value_info('x_i', TensorProto.FLOAT, [1])
state_out = make_tensor_value_info('state_out', TensorProto.FLOAT, [1])
scan_out = make_tensor_value_info('scan_out', TensorProto.FLOAT, [1])

add_node = make_node('Add', ['state_in', 'x_i'], ['state_out'])
identity_node = make_node('Identity', ['state_out'], ['scan_out'])

body = make_graph(
    [add_node, identity_node],
    'scan_body',
    [state_in, x_i],
    [state_out, scan_out])

# Main graph
init_state = from_array(np.array([0.0], dtype=np.float32), name='init_state')

X_scan = make_tensor_value_info('X', TensorProto.FLOAT, [None, 1])
Y_final_state = make_tensor_value_info('final_state', TensorProto.FLOAT, [1])
Y_cumsum = make_tensor_value_info('cumsum', TensorProto.FLOAT, [None, 1])

scan_node = make_node(
    'Scan',
    ['init_state', 'X'],
    ['final_state', 'cumsum'],
    body=body,
    num_scan_inputs=1)

graph_scan = make_graph(
    [scan_node], 'scan_cumsum',
    [X_scan], [Y_final_state, Y_cumsum],
    [init_state])

model_scan = make_model(graph_scan, opset_imports=[make_opsetid('', 15)])
model_scan.ir_version = 8

sess_scan = InferenceSession(model_scan.SerializeToString(),
                             providers=['CPUExecutionProvider'])

# Test
x_data = np.array([[1], [2], [3], [4]], dtype=np.float32)
final_state, cumsum = sess_scan.run(None, {'X': x_data})

print(f'Input X:        {x_data.flatten()}')
print(f'Cumulative sum: {cumsum.flatten()}')
print(f'Final state:    {final_state.flatten()}')
print(f'Expected:       {np.cumsum(x_data.flatten())}')
print(f'Match: {np.allclose(cumsum.flatten(), np.cumsum(x_data.flatten()))}')

<a id='section-6'></a>
## Section 6: The Loop Operator

### Formal Semantics

The `Loop` operator combines **for** and **while** semantics:

$$\text{For } i = 0, 1, \ldots:$$
$$c_{i+1}, s_{i+1}, y_i = f_{\text{body}}(i, c_i, s_i)$$
$$\text{Stop when } c_{i+1} = \text{false} \text{ OR } i \geq M$$

### Body Subgraph Interface

The body subgraph receives:

| Input | Type | Description |
|-------|------|-------------|
| `i` | `int64` scalar | Current iteration number |
| `cond` | `bool` scalar | Continue condition |
| `state_0..N` | `tensor(T)` | Loop-carried state variables |

The body subgraph produces:

| Output | Type | Description |
|--------|------|-------------|
| `cond_out` | `bool` scalar | Whether to continue (while condition) |
| `state_0..N` | `tensor(T)` | Updated state variables |
| `scan_0..K` | `tensor(T)` | Values to accumulate (optional) |

### Loop Modes

```
Mode 1: FOR loop (fixed iterations)
  M = max_iterations, cond_init = true, body always returns true
  → Runs exactly M times

Mode 2: WHILE loop (condition-based)
  M = max_int, cond_init = true, body returns data-dependent cond
  → Runs until condition becomes false

Mode 3: FOR-WHILE (bounded while)
  M = max_iterations, body returns data-dependent cond
  → Runs until condition becomes false OR M iterations
```

### Example: Fibonacci with Loop

The Fibonacci recurrence $F_{n+1} = F_n + F_{n-1}$ is a natural fit for the Loop operator, as it requires carrying state (the previous two values) across iterations:

$$\begin{align}
s_0 &= (F_0, F_1) = (0, 1) \\
s_{i+1} &= (s_i[1], s_i[0] + s_i[1]) = (F_i, F_{i+1})
\end{align}$$

<a id='section-7'></a>
## Section 7: Control Flow vs Vectorized Operations

### When to Avoid Control Flow

Control flow operators are powerful but expensive. They prevent many graph-level optimizations and add per-iteration overhead. **Always prefer vectorized operations when possible**.

| Avoid (Control Flow) | Prefer (Vectorized) | Reason |
|:---|:---|:---|
| `If` for element-wise conditions | `Where` operator | `Where` processes all elements in parallel |
| `Scan` for matrix operations | `MatMul`, `Conv` | Batch operations are hardware-optimized |
| `Loop` with fixed iteration count | Unrolled graph | Eliminates loop overhead |
| `Scan` for prefix sum | `CumSum` operator | Dedicated operator with optimized kernel |

### Performance Impact

```
Vectorized (Where):       Control Flow (If per element):

  ┌──────────────────┐    ┌──────────────────────────────┐
  │ X ──▶ Where ──▶ Y│    │ for each x in X:             │
  │    (single op,   │    │   if x > 0:                  │
  │     GPU-parallel)│    │     y = x                    │
  └──────────────────┘    │   else:                      │
                          │     y = 0                    │
  Time: O(1) kernel       │                              │
  launches                │ Time: O(N) kernel launches   │
                          └──────────────────────────────┘
```

### The `Where` Operator (Vectorized Conditional)

The `Where` operator is the vectorized alternative to element-wise `If`:

$$\text{Where}(C, X, Y)_i = \begin{cases} X_i & \text{if } C_i = \text{true} \\ Y_i & \text{if } C_i = \text{false} \end{cases}$$

In [ ]:
# Demonstrate Where as a vectorized alternative to If

# Build a model: Y = Where(X > 0, X, 0)
# This is equivalent to ReLU, but demonstrates the pattern

X_w = make_tensor_value_info('X', TensorProto.FLOAT, [None])
Y_w = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

zero_init = from_array(np.array([0.0], dtype=np.float32), name='zero')

greater_node = make_node('Greater', ['X', 'zero'], ['cond'])
where_node = make_node('Where', ['cond', 'X', 'zero'], ['Y'])

graph_where = make_graph(
    [greater_node, where_node],
    'where_demo', [X_w], [Y_w], [zero_init])

model_where = make_model(graph_where, opset_imports=[make_opsetid('', 15)])
model_where.ir_version = 8

sess_where = InferenceSession(model_where.SerializeToString(),
                              providers=['CPUExecutionProvider'])

x = np.array([-2, -1, 0, 1, 2, 3], dtype=np.float32)
result = sess_where.run(None, {'X': x})[0]

print(f'Input X:  {x}')
print(f'Where(X>0, X, 0): {result}')
print(f'Expected (ReLU):  {np.maximum(x, 0)}')
print(f'Match: {np.allclose(result, np.maximum(x, 0))}')

<a id='section-8'></a>
## Section 8: Formal Semantics

### Subgraph Variable Scoping

Subgraphs in ONNX follow **lexical scoping** — they can reference tensors from any enclosing scope:

$$\text{scope}(G_{\text{sub}}) = \text{locals}(G_{\text{sub}}) \cup \text{scope}(G_{\text{parent}})$$

This means a then-branch subgraph can use tensors computed in the main graph without explicitly passing them as inputs. The runtime resolves names by walking up the scope chain.

### Scan as a Higher-Order Function

The Scan operator is essentially a **fold** (left reduction) combined with a **map**:

$$\text{Scan}(s_0, [x_0, \ldots, x_{T-1}], f) = \left(s_T, [y_0, \ldots, y_{T-1}]\right)$$

where:
$$s_t, y_{t-1} = f(s_{t-1}, x_{t-1}) \quad \text{for } t = 1, \ldots, T$$

This is equivalent to Python's `itertools.accumulate` or Haskell's `scanl'`.

### Loop Invariant

For the Loop operator, the **loop invariant** is:

$$\forall i \in [0, M): \text{type}(s_i) = \text{type}(s_0) \wedge \text{shape}(s_i) = \text{shape}(s_0)$$

State variables must maintain their type and shape across all iterations. This constraint enables static memory allocation.

### Termination Guarantee

The `Loop` operator guarantees termination through the maximum iteration count $M$:

$$\text{iterations} = \min\{M, \min\{i \mid c_i = \text{false}\}\}$$

Setting $M = 0$ creates a no-op; setting $M = \text{INT64\_MAX}$ creates a pure while loop.

In [ ]:
# Summary visualization: control flow operator comparison
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

ops = [
    ('If', 'Conditional\nexecution', 'cond → A or B',
     'Routing, early exit,\nspecialized paths', '#FF9999'),
    ('Scan', 'Row-wise\niteration', 's, x_t → s, y_t',
     'RNN steps, cumulative\nreductions, sequences', '#99FF99'),
    ('Loop', 'For/while\nloop', 'i, cond, s → cond, s, y',
     'Dynamic length decoding,\nconvergence loops', '#9999FF'),
    ('Where', 'Vectorized\nconditional', 'cond, X, Y → Z',
     'Element-wise selection\n(prefer over If)', '#FFFF99'),
]

for i, (name, desc, sig, use_case, color) in enumerate(ops):
    x_pos = i * 3 + 0.5
    bbox = dict(boxstyle='round,pad=0.5', facecolor=color,
                edgecolor='black', linewidth=1.5)
    ax.text(x_pos, 4, name, ha='center', va='center',
            fontsize=14, fontweight='bold', bbox=bbox)
    ax.text(x_pos, 3, desc, ha='center', va='center', fontsize=9)
    ax.text(x_pos, 2, sig, ha='center', va='center',
            fontsize=8, family='monospace',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.text(x_pos, 0.8, use_case, ha='center', va='center', fontsize=8)

ax.set_xlim(-0.5, 12.5)
ax.set_ylim(0, 5.5)
ax.set_title('ONNX Control Flow Operators — Comparison',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

<a id='section-9'></a>
## Section 9: Key Takeaways & Interview Questions

### Summary

| Concept | Key Point |
|---------|----------|
| **Subgraphs** | `GraphProto` objects embedded as node attributes, enabling control flow in static DAGs |
| **If** | Conditional execution of one of two subgraphs based on a boolean scalar |
| **Scan** | Row-wise iteration with carried state and accumulated outputs |
| **Loop** | Combined for/while with iteration count and condition |
| **Where** | Vectorized element-wise conditional — always prefer over `If` for element-wise operations |

### Critical Rules

1. **Both `If` branches must match**: Same number of outputs with compatible types
2. **Scan state shapes are fixed**: State tensors cannot change shape across iterations
3. **Loop always terminates**: Maximum iteration count $M$ provides an upper bound
4. **Prefer vectorized ops**: `Where`, `CumSum`, batched operations over control flow
5. **Subgraphs inherit scope**: Can reference tensors from parent graphs

### Interview Questions

1. **Q**: How does ONNX implement control flow in a static DAG?
   - **A**: Through operators (`If`, `Loop`, `Scan`) that take `GraphProto` objects as attributes. The control flow operator manages the execution of these embedded subgraphs.

2. **Q**: When should you use `Where` instead of `If`?
   - **A**: Always for element-wise conditions. `Where` processes all elements in a single parallel kernel, while `If` with subgraphs adds per-decision overhead and prevents vectorization.

3. **Q**: What is the formal semantics of the Scan operator?
   - **A**: Scan is a fold+map: it iterates over rows of input tensors, carrying state $s_t$ across iterations via $s_{t+1}, y_t = f(s_t, x_t)$, and accumulates the scan outputs $[y_0, \ldots, y_{T-1}]$.

4. **Q**: What guarantees termination of the Loop operator?
   - **A**: The maximum iteration count $M$ provides an upper bound: $\text{iterations} = \min(M, \min\{i \mid c_i = \text{false}\})$. Setting $M=0$ creates a no-op.

---

**Next:** [Functions](../06_Functions/) — Define reusable operator combinations.